<h3 style="color:#6FA8DC; font-weight:bold">Multivariate Imputation → Iterative Imputer</h3>

### What is Iterative Imputation?
Iterative Imputation predicts missing values using **other features**.

Instead of simply taking a mean, one feature is treated as the target and a model predicts its missing values from the other features. This is repeated for the missing columns.

```text
Initial values
      ↓
Fill missing values temporarily
      ↓
Predict missing values of Feature 1
      ↓
Predict missing values of Feature 2
      ↓
Predict missing values of Feature 3
      ↓
Repeat iterations
      ↓
Final imputed dataset
```

This is also commonly associated with **MICE (Multiple Imputation by Chained Equations)** style thinking.


### Why use it?
Use Iterative Imputer when numerical features have useful relationships and you want a model-based estimate.

**Advantages**
- Uses relationships between features.
- More flexible than mean/median.
- Can use different estimators.

**Disadvantages**
- More computationally expensive.
- Can overfit if not configured carefully.
- Results depend on the estimator and parameters.
- Still needs careful train/test handling.

### Production note
Fit the imputer only on training data and put it inside a Pipeline.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge, LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

df = pd.read_csv('train(1).csv')[['Age','Pclass','Fare','Survived']]
df.head()

In [ ]:
X = df.drop(columns=['Survived'])
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=2
)

In [ ]:
iter_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('imputer', IterativeImputer(
        estimator=BayesianRidge(),
        max_iter=10,
        random_state=42
    )),
    ('model', LogisticRegression(max_iter=1000))
])

iter_pipe.fit(X_train, y_train)
y_pred = iter_pipe.predict(X_test)

accuracy_score(y_test, y_pred)

### Important parameters
- `estimator` → model used to predict missing values.
- `max_iter` → number of imputation rounds.
- `initial_strategy` → initial temporary filling strategy.
- `random_state` → reproducibility.

### KNN vs Iterative
| KNN | Iterative |
|---|---|
| Uses nearest rows | Uses predictive models |
| Distance-based | Model-based |
| Needs attention to scaling | Scaling depends on estimator |
| `n_neighbors` important | `estimator`, `max_iter` important |

**Revision:** Iterative Imputer repeatedly predicts missing values from the other features.